Investigated settings

Done things 
    EEG
        LDA
        SVC
        RandomForest
    MEG
        LDA
        SVC
        RandomForest
        
    EEG+MEG
        RandomForest
        SVC
        LDA

Interesting_right =   [5, 33, 45, 49]<br>
Interesting_left =   [4, 32, 44, 48]


Sebastien Velut

Observations on spectrograms(see Visualization notebook):  <br>
    1. in every case (varying classifiers) Not personalized is better (or equivalent) to Personalized <br>
    2. For both SVM and RandomForest, NROIs Performance increase with NROIs, but starts to saturate already at 10 ROIs. For LDA can vary <br>
    3. SVM and RandomForest better than LDA in EEG and MEG alone. Improvement present but lower when using EEG concat MEG <br>
    4. both for SVM and RandomForest, EEG concat MEG perform as the best between MEG and EEG alone. For LDA, it can be better than the best of the 2 <br>

# Brainstormings - Idee

Todo:
- Fai classificazione con tutte le regioni; a posteriori di SVM controlla quali sono le più significative (T test su differenza tra MI e Rest)
- Prova test statitsico differente da T test per la scelta delle regioni e.g. Energy Distance Test / k-sample E-test / T test in corrispondenza solo della frq che ha il max diff Re-MI
- verifica come si comporta un MLP per feature estratte a mano (per avere una stima di che succederà con DeepLearning. In realtà non é detto che debba usare come calssificatore dopo le features extraction un MLP.... potrei usare RandomForest?? vediamo...)

Incontro 04/12/2025:
- Per utilizzare Deep Learning, potrebbe essere utile selezionare solo alcune regioni!
- Criterio per selezione regioni:
    ◊ investiga meglio cosa succede per il nostro criterio: "cross validation of region selection", se la selezione é significativa, pero ogni soggetto, anche al variare dei trials (split test train kfold) devo ottenere stessa selezione di regioni!
    ◊ DEEP Learning: trovare un buon criterio per fare in modo di selezionare regioni significative e semplificare il lavoro all'autoencoder! 
        - potremmo utilizzare un T-test, con  ipotesi che per ogni regione, le distribuzioni di dati MI - Rest sono differenti.
        - 68 * 250 = 17.000 é troppo! dobbiamo scegliere per bene le regioni! (aiutati con risultati di MLP su features a mano, forse?)
- Auto encoder evaluation: The average Pearson correlation coefficient between the reconstructed signals and the original signals for all 1-second segments in the test set was 0.939


### Brainstorming 5-12 Dec

- !!! Problema !!!
    ◊ Sto spingendo a studiare un criterio per la selezione delle regioni. Ma la selezione deve essere di GRUPPO o di SOGGETTO ??
- Se devo fare selezione di gruppo, come faccio? Raramente ho stesse regioni

- Domanda:<br>
    ◊ Quanto sono importanti le altre regioni per la struttura di una singola regione?<br>
        (stavo pensando che potrei addestrare un unico autoencoder sulla singola coppia EEG-MEG di una stessa regione)<br>
        (e sfruttare dati di tutti i pazienti e forse di tutte le regioni???)<br>
        (se sfrutto dati di tutte le regioni, posso assegnare un significato alla struttura dell'encoded layer, e poi quando addestro classificatore, posso usare conv layers!!!)<br>
            
            regioni
            0 1 2 3 4 5
        f 0 . . . . . .
        e 1 . . . . . .
        a 2 . . . . . .
        t 3 . . . . . . 


        (o anche solo su sottogruppi di regioni che sono appartenenti a stessa zona del cervello??? (diminuisco features))
    ◊ Successivamente potrei selezionare regioni in base a T test su encoded layer! (Baseline vs MI) o verificare cosa succede prendendo tot reigoni o escludendo tot regioni

    
LOO Leave One Out
- Idea: potrei addestrare singolo autoencoder (PolarCAP setting) su tutte regioni, a singola regione (1 EEG. per volta) (tutti soggetti, o LOO) - INTERSUBJECT
- Poi potrei, per scelta di regioni, fatta su singolo sogetto, prendere le tot regioni che massimizzano la performance del soggetto 
- Cosa bella: Potrei addestrare su tutti i dati di una sessione precedente e poi FineTuning (frozen or not) su seesione successiva 

Varie pipelines:<br>
fare tutto prima su singolo soggetto tutte regioni (subj buono, il 2)<br>
windows di 0.5 sec, sovrapposizione di 0.25 sec,che indagano i trials da 3 a 6 secondi, dove devono fare il task<br>
(tirials x windows_per_trial x regions) = (96x12x68) = 70.000 samples
- autoencoder, very easy, su timeseries (forse anche CNN semplice, senza struttura autoencoder)
- invece di timeseries "raw", diverse bande (tipo VAEEG paper), "multiscale" CNN
- vedere se invece di timeseries i powerspectra
- IDEA SIMPATICA MA NON SO SE FATTIBILE: concatenare al bootleneck features estratte a mano, verificare con intepretability tools quale features sono più significative

### Brainstormig 08/01/2026

Come addestro autoencoder:		<br>
(0.5sec time series, 0.25s overlap, entire window 3s) =	<br>	
11 windows		<br>
96 trials		<br>
20 soggetti		<br>
68 regioni		<br>

| tipology           | region type       | specificity       | subjects        | N. of examples (factors) | N. of examples  | feasible (based on N. examples)?|
|--------------------|-------------------|-------------------|-----------------|---------------------------|----------------|- |
| 1 Rete      | single region     | region agnostic   | all subjects    | 11 x 96 x 20 x 68         | 1.436.160      |✓ |
| 1 Rete      | single region     | region agnostic   | single subject  | 11 x 96 x 1 x 68          | 71.808         |✓ |
| 68 Reti     | single region     | region specific   | all subjects    | 11 x 96 x 20 x 1          | 21.120         |- |
| 68 Reti     | single region     | region specific   | single subject  | 11 x 96 x 1 x 1           | 1.056          |- |
| 1 Rete      | multiple regions  | —                 | all subjects    | 11 x 96 x 20 x 1          | 21.120         |- |
| 1 Rete      | multiple regions  | —                 | single subject  | 11 x 96 x 1 x 1           | 1.056          |X |




Domanda:
- è necessario fisiologicamente scegliere le stesse regioni EEG e MEG?
    è possibile che una data attività cerebrale sia evidente in alcune regioni nell'EEG e in altre nella MEG?
    se devo scegliere stesse regioni, dobbiamo pensare a come fare
- Intensità del segnale è significativa (sospetto di si)
    in base a questo scegliere corretta normalizzazione

### Selezione unica di regioni a priori:
1. 
    - Scelgo un criterio di selezione (Cohen's d); 
    - Seleziono le regioni su singolo soggetto (Considero le prime TOT regioni per ogni soggetto) ; 
    - Seleziono le regioni selezionate per più soggetti
2. 
    - Scelgo un criterio di selezione (Cohen's d); 
    - Seleziono le regioni su singolo soggetto (Considero le prime TOT regioni per ogni soggetto) ; 
    - Seleziono le regioni che hanno posizione media (tra soggetti) più bassa

Entrambi i casi sembrano dare stessi risultati (vedi notebook RegionSelection -> Cohen -> GlobalSelection)

### 12/01/2026
Idea: un encoder-decoder che da MEG faccia passare a EEG (o (e??) viceversa)



### Brainstorming Incontro 26/01/2026

FUSIONE a (attenzione a come scegliere regioni):
- solo regioni interessanti
- cohen's: prime 5 tra MEG, prime 5 EEG e "accorparle"
- "le migliori" a posteriori dopo aver visto le performance su singola regione

###  Auto-Brainstorming 28/01/2026
Come accorpare dopo la features extraction - caso autoencoder single region, region agnostic (vedi problemi concettuali)

### 09/02/2026

Encoder_timeseries_Decoder_PowerSpectra??

### 26/02/2026

Brainstorming con Marie:
- Prova a fare 3 normalizzazioni per decoder to PowerSpectra: (normalizzazione classica per regione; normalizzazione per banda; normalizzazione per bin di frequenza)

- OPPURE! non lavorare con i power spectra, ma con il loro logaritmo (così do maggiore importanza al peso delle alte freqs, ma evito di fare normalizzazioni strane)

### 27/02/2026

- Introduci GRU layer da qualche parte dopo ultimo CONV in encoder

valuta aggiungere:
- residual blocks
- dilated convolution
- multi-receptive field

# Pre-Deep Learning
Initial Investigations / Region selection criteria (Cohen, Global vs Subject specific...)

## 12/01/2026
Da ora:
- Usa 5 Folds (ho verificato non cambia con 10) 
- Concentrati su Cohen's d effect size (altri sono inferiori come performances e come significato fisiologico)


## 05 - 08 / 12 / 2025

- cerco di capire come visualizzare la robustezza di un method di selezione delle features al variare dei kfold
- grafico hist2d region vs rank in all folds (per ora sul criterio mio, ma dattabile a tutti i criteri tipo T-Test)
![BoG2](Ranking_Regions_basic.png)

- ho trovato un modo per fare il sort delle regioni, in modo da mettere in evidenza quelle più significative
![BoG](Ranking_Regions_sorted.png)


## 08 - 09/12/2025

- Un trovato un T test multivariato su pingouin (multivariate_ttest)
- Ho effettuato il T test: verifico l'ipotesi statistica per ogni soggetto, per ogni regione, per ogni fold, che la popolazione di Power specra di Rest abbia media diversa da MI (popolazioni approssimate da gaussiane)

## 10/12/2025

- automatizzo plot visualizzazione scelta regioni
- Confronto tra robustezza di T test e MyCriteria per selezionare regioni. MyCriteria é più Robusto tra i fold (trend coerente tra i soggetti)

## 11/12/2025

- Indago meglio su come si comportano i critieri di scelta delle regioni
- Noto che la distribuzione dei p-value su multi T test e abbastabza uniforme in scala log. variabilità da 10^-14 a 10^-1
- Alcune volte il MyCriteria scelglie di guardare frequenze troppo basse, (prima davo spettro da 2Hz in poi)
- Si risolve dando spettro in 4-30 Hz
- Rimane il problema che alcune volte focus su differenze grandi, ma non significative (ricadono all'inizio della banda di freq, vedi Img dopo, in questo caso selezionavo auesta ROI come prima) ◊ ATTENTO non significativa perchè ho notato che Rest ha varianza enorme! e anche perchè qui
![BoG2](PowerSpectra_sub5_ROI33.png)
- Ho visualizzato tutti i PowerSpectra delle differenti regioni del soggetto 4. Non ci sembra essere una differenza elevata tra MI e Rest. Come mai classifico bene?? Che sta succedendo??

## 12/12/2025


Ho fatto vedere a Marie
- visualizzazione robustezza (multi T test vs Mycriteria)
- corrispondenza con performances (no a priori regions)
- Power spectra regioni soggetto 4

Abbiamo osservato che:
- per il MyCriteria test sto facendo average su trials, average (selection) su frequency. ho molto più averaging
- per il multi T test su tutto il power spectra per come lo ho fatto, per ogni reigone ho "variabilità" su tanti samples e tanti trials
- il "molto più averaging" del MyCriteria, è molto probabile che sia questo a renderlo più robusto

- Fai t test su media su freq alpha e beta
- Vedere performances con questo nuovo multi (2 features) T test

- Ho fatto MyCriteria su 8 - 30 Hz. Stessa robustezza e Funiziona meglio effettivamente ! (ho controllato visivamente gli spectra delle regioni scelte e. sembra ok, migliore)
- Ho fatto multi(2 valori) T test su media su freq alpha e beta, ho notato che è più robusto rispetto multi T test su tutto vettore del power spectra, ma è sempore meno robusto del MyCriteria
- Oltre a robustezza, bisogna capire qunto sia buono un method di selezione. Visualizzo se sto selezionando regioni significative, ma non sempre riesco a dare interpretazione esaustiva. Provo a verificare bontà di selezione a posteriori, in base a quanto performano i miei classifier

## 15/12/2025 Incontro con tutti


- Provo a verificare bontà di selezione a posteriori, in base a quanto performano i miei classifier.
- A questo scopo, check delle performance dei metodi di selezione "MyCriteria" e "T test su mean alpha beta", al variare di classifier, al variare di DataType.




Done:
- New criteria - effect size (verifica robustezza, performances, come per gli altri criteri)
- Verifica comunione di regioni tra soggetti tra le prime 20 regioni scelte (al variare dei criteri) (Images-Brain-Region_selection...)
- prova ad aumentare i kfold (RepeatedStratifiedKFold di scikit-learn)

## 17/12/2025


- testo il Choen_effect_size
- verifico la robustezza e scopro che sembra leggermente meno robusto sia del MyCriteria che del T test on mean alpha e beta
- verifico le performances e scopro che sono migliori (poco) sia del MyCriteria che del T test on mean alpha e beta

- Verifico se c'é sovrapposizione tra i soggetti nelle regioni scelte per cohen's d effect size.
- effettivamente (20 best regioni) molti scelgono quelle MOTORIE! (left hemi)

## 18/12/2025


Fatto (tutto contenuto nelle immagini BrainPlot):
- salva i plot dei brain (selection dei primi 20)
- controlla che succede con gli altri criteri
- controlla se almeno 1 delle regioni scelte tra le 20, per i soggetti, rientra in una di quelle motorie (left hemi)
- Controlla se la selezione la faccio calcolando gli spettri solamente dal secondo 3 in poi (Non sembra stabile quanto gli altri, e le performace sembrano peggio???)
- prova ad aumentare i kfold (RepeatedStratifiedKFold di scikit-learn)



## 19/12/2025 - Incontro con Marie

Done:
- MyCriteria: Dividi per il rest dello stesso bin di dove hai max(rest-MI) e vedi che succede (battezzato MyCriteria2)
- verify my effect size computation is the same of the pingoun one (pingouin effect size) (Paired=True) (verified it is Ok)
- verifica a quale frequenza corrisponde max (In Cohen's d Effect size, in order to have control on frequencies among folds, subjects etc)


## 23-29/12/2025

### 23/12/2025
- Verified my effect size computation is the same of the pingoun one (pingouin effect size) (Paired=True) (verified it is Ok, see Test notebook)
- Organized github folder 
- Migration on Salerno Workstation (environments, adaptation of codes, downloading data etc...)
- Compatibility between laptopts - Workstation (now I can work from every device)

### 24/12/2025
- Training with new version of MyCriteria selection (now not dividing for the max of rest, but for the value of the rest where i found the max(MI-Re))
- It is not better in performance than the normal MyCriteria
- Not more robusts

### 25/12/2025
- Sistemato il codice per fare i violin plot di tutti i soggetti al variare delle impostazioni


- Violin Plot - fixed classifier, method selection, varying Nrois, Datatype. 
- Observation: The more the ROIs, the better the performance (Sempre per EEG concat MEG) (SVC, RandomFoest; for each selection criterion; LDA solito comportamento strano)
- Bisogna capire fino a dove è significativa la differenza



### 26/12/2025
- provo ad incrementare i Kfold - 10 repeated stratified kfolds, "BCI_Performances_region_selection_on_repeated_10folds_8_30_Hz"
- Testato per MyCriteria, Cohen's d effect size, perogni classifier, per EEG, MEG, EEG concat MEG

### 27/12/2025
- Verifico che con 10 k-folds le performanes non sono variate
- grafico in maniera diversa, tenendo traccia di ciascun soggetto. 3 Violin plot per soggetto

### 28/12/2025
- verifico a quale bin di freq.z corrisponde il max di cohen's d effect size
- plot al variare dei soggetti, per le n_best regioni scelte

### 29/12/2025 - ...
- Lavoro sul paper di Mario; Inizio sistemando le figure; Trovo inconsistenze per NeuralNetworks

## 07/01/2026
Todo - by Silvia:
- Regions_Selection.ipynb -> Cohen's d -> Verify Frequency bin selection -> Controlla non solo dove è il massimo, ma potrei plottare la media (su regioni? Normalizzo?) di Cohen's d al variare del bin di frequenza


Done:
- fai un altro pannello di performance: Scelgo le 3 (5,10,...) migliori in "generale" su tutti i soggetti e vedo che succede alle performances in base a questo criterio di selezione generale, le stesse regioni per tutti i soggetti (controlla sempre se la 44, 32 etc sono scelte)
- append ai files di region selection le selezioni fatte senza dividere per kfold

## 08/01/2026 - 10/01/2026

Incontro con Marie<br>
Todo - by Gio:
- Se hai tempo, prova a vedere performance selezionando 3 regioni, al variare di tutte le combinazioni di 3 regioni, magari selezionare tra le regioni che performano meglio a singola regione (Visualization.ipynb -> ## Regions' Investigation ) 

Done:
- Criterio per selezione unica di regioni per tutti i soggetti (in vista di costruire un autoencoder a più regioni). 
    - Seleziona le TOT_1 regioni che vengono più scelte in generale per la selezione di singolo soggetto, e verifica che ci siano almeno le TOT_2 migliori reioni per soggetto considerate per tutti i soggetti.
    - O almeno grafica quante tra le migliori TOT_2 regioni di singolo soggetto sono selezionate tra le TOT_1 generali. 
    - Verifica poi quali sono le performances facendo una selezione unica per tutti i soggetti.
    - Dai uno sguardo se le regioni importanti left [4, 32, 44, 48] vengono incluse

- Working on global region selection (Regions_Selection.ipynb->Cohen->GlobalSelection)
    - found that (for Cohen based global selection) considering the best 15 for all subjects, and then taking the 10 most selected, we can retrieve also regions 32,44,48, both for MEG than EEG (poor region 4, it is not selected) 
    - the 10 most selected does not completely overlap between EEG and MEG! (what can I do?)

- Verifying performances on global selection (MyCriteria and Cohen): 
    saving in /Results/BCI_Performances_region_selection_on_repeated_10folds_8_30_Hz.pkl
- Comparison with subject-based selection (Visualization->Starting on 12 / 2025->Plot ALl subjects -> DIfferent Criteria). 
    Scopro che la scelta globale non è meglio.
    (LDA indifferente, SVC miglioramenti minimi per MyCriteria peggioramenti piccoli per Cohen, RandomForest peggioramenti Cohen EEG!)

## 12/01/2026 - Incontro con tutti
brainstorming (see above) - Encoder-Decoder EEG->MEG (ViceVersa)


## 13-14/01/2026
incontro marie: ho fatto
- $\forall$ subject, plotted of 3 (5,10,20) first selected regions on brains and take track of performances (svc)
    - Inserito nei plot performance che avremmo se abbiabo selezione globale (e fai anche i plot delle regioni scelte globalmente)
    - passa i plot; Lista soggetti non performanti a Marie
    - overlap tra EEG e MEG (globale)

# DEEP LEARNING HAS BEGUN

## 15-17/01/2026
Ho Dimenticato di aggiornare il diaro! 
- Inizio a Costruire prima classe che contempli rete CNN, autoencoder, PolarCAP architecture. Expandibile a variational autoencoder - adversarial autoencoder
- Lavoro al paper di Mario  

## 19-20/01/2026
- lavoro da Silvia Dell Pc
- risolto problemi di tensorflow
- risolto con fatica (ticket icm support) problemi di connessione VPN da icm
- continuo a sviluppare classe
- lavoro paper di Mario 
- Visualizzo std: axis tempo, average on trials, per region, per subject (Inspection -> Verify amplitude of signals)

## 22/01/2026
incontro con marie - brainstorming:
- normalizzazione per regione non canale.
- FInestra temporale da considerare (training) 3-6 s. potremmo in futuro considerare anche 1-3 (task c'è me non feedback) (inserisco nel training? testing??) possiamo verificare se i soggetti stavano facend oil task anche prima del sec 3
- plot per ciascuna finestra temporale (vs time) stima accuracy. per vedere come va accuracy vs time

## 21-23/01/2026
Lavoro su SUBJECT 2 
creo functions/classi per fare:
- Split; Windowing; Normalization 
<br>
<br>
- Split Train Val Test: 
    - Windows del Singolo trial rimangono sempre tutte nel training o validation o test
    - split preserva balancing di regioni e di MI vs Rest

- Normalizzo ciascuna window per z-score "globale di reigone":
    - std PER REGIONE: calcolata come std di tute le windows di quella regione;
    - lo stesso per mean
    - motivo è: potrei avere che amplitute è significativa in MI vs Rest


Verifico normalizzazione (Test_DL.ipynb -> In Utils... -> ALL together).<br->
Noto che 
- In alcune regioni ho sensibile differenza in medie di std e means tra MI e Rest, in altre no 
- alcune volte i valori delle tracce normalizzate con "region wise norm" sono hanno range elevati(vedi)
- se normalizzo "trace-wise" succede che comunque i max(abs(trace)) raggiungono valori di 3 ... quindi ok, normalizzare region wise non è assurdo


## 24-26/01/2026
Inizio i primi addestramenti deep! (rete classificazone CNN easy) 
- implemento addestramento su 5 folds
- Computed performances al variare delle regioni. Confronto risultati di CNN per singola regione con quello che succedeva con power-spectra features + SVM (randomforest) per singola regione
- rifatta analisi classica (powerspectra+SVM) ma solo con EEG, solo con MEG, both $\forall$ classifiers



## Incontro Silvia e Marie

TODO:
- Usare stesso classifier per Features power-spectra vs Features di CNN
- combinazione regioni e confronto alla pari tra due features
- FUSIONE a (attenzione a come scegliere regioni):
    - solo regioni interessanti
    - cohen's: prime 5 tra MEG, prime 5 EEG e "accorparle" (quindi dare EEG anche a regioni scelte da MEG & viceversa)
    - "le migliori" a posteriori dopo aver visto le performance su singola regione
- addestrare MLP per singola regione PER SINGOLO SOGGETTO - power spectra features


## 27/01/2026

- Uniformo dataframe di performances tra DL e classico

TOOD: IMPORTANTE
- encoded Layer, now i used convolutional layer in the encoded. But im this way i lose a big number of parameters just before the encoded Layer.
- Try to use a locally connected layer (WHY KERAS DELETED IT??)
- Costruisco codice per MLP, sarà addestrato su hand extracted features (che poi confronterò con features CNN extracted)


## 28/01/2026
- Ho costruito pipelines per introdurre MLP come calssifier su power-spectra
- Brainstorming - architettura post encoded x confronto diretto con quello fatto sino ad ora
- Provato ad addestrare MLP per singola regione. Impiega troppo (68 regioni x 20 subj x 5 folds...)
- Lavoro paper mario un po'
- Cambio Comments in Region Selection



## 29/01/2026
- Procedo con Cohen's Effect size selection:
    - Hand-designed features (power spectra): ora addestro MLP (Non Sono riuscito, impiega troppo tempo)
    - Errore server unisa!

## 30/01/2026
Scopro esiste leakage molto subdolo tra training e validation: leakage di diverse regioni su stesso trial:<br>

dataset: = (n_trials, n_regioni, timepoints) # Ogni "dato elementare è un vettore di lunghezza timepoints, organizzato in una matrice (n_trials x n_regioni)
- prima 
    - inserivo in una "lista" ogni trial data.shape = (n_trials*n_regioni, timepoints)
    - assegnavo un label (Rest/MI) e una regione ad ogni trial
    - dividevo train/val/test set, bilanciando 1.label, 2.regioni
    - dividevo ogni traccia (sia del training/val/test) in windows di 0.5s
    - in questo modo:
        - bilanciavo label, regioni negli split 
        - facevo in modo che windows di stesso trial non andassero sia in training che validation, ma solo in un set

- ora
    - prima di inserire in una "lista" i trials, divido già in train/val/test set, sempre bilanciando 1.label, 2.regioni
    - per ogni set, inserisco in "lista" ogni trial corrispettivo  data_training.shape = (n_trials_training, n_regioni, timepoints)
    - divido ogni traccia (sia del training/val/test) in windows di 0.5s
    - in questo modo continuo a:
        - bilanciare label, regioni negli split 
        - fare in modo che windows di stesso trial non andassero sia in training che validation, ma solo in un set
    - ma ora ho anche:
        - sicurezza che regioni differenti di stesso trial non vadano in set diversi

Con il nuovo modo non solo le performance sono peggiori (per validation/test set), ma noto anche che l'algoritmo "non sta imparando molto" <br>
Non esiste mai una diminuzione di loss del validation set! overfit ?!<br>
Esistono quindi caratteristiche comuni tra diverse regioni per lo stesso trial che la rete impara, a discapito del compito vero


NOTO anche cose buone: Performances sono migliori per le regioni interessanti

## 31/01/2026
- Ho addestrato autoencoder 1 region, region agnostic. Noto:
    - Riprodue dati bene, ma filtrati lowpass a 20/22 Hz


- Ipotizzo che "poor results" di prima possano essere dovuti al fatto che ci siano features troppo diverse tra differenti regioni
- Addestro autoencoder 1 region, region specific, per vedere se qusta limitazione deriva da troppa variabilità tra tante regioni
    - scelgo global selected da cohen
    - Noto che non miglioro significativamente (a primo sguardo)

## 02/02/2026
Incontro tutti, DONE:
- Addestra CNN semplice MA solo su regioni selected (per ora slezione su regioni importanti) e vere cosa succede
    - ‘Accorpare le regioni’ e addestra classifier
    



DONE: 
- Provo ad addestrare CNN easy su singola regione, e confrontare con pipeline SVM RandomForest Normale. Scelgo per ora di addestrare su regioni interesting right = [5, 33, 45, 49]     left = [4, 32, 44, 48]
    - Da RIFARE ?? Mi vengono risultati strani (val loss non scende)
    - (2026-02-02-16_14_31_CNN_1region_regionspecific_Interesting_regions E 2026-02-05-12_02_47_CNN_1region_regionspecific_Interesting_regions)
- Working on Trainings_DL - reigon agnostic - autoencoder
- Autoencoder: Upsample (scipy resample) to double the samples of the signal: maybe add another layer after this procedure.
    - Fatto, Ma non noto nssun miglioramento nel riprodurre alte frequenze

## 03-04/02/2026
- Addestro classfier di regioni accorpate (sceglo di lavorare su Interesting regions) (BCI_Performances_DNN.pkl)
    - 2026-01-30-18_41_52_Autoencoder_1region_region_agnostic_First_Try
    - 2026-02-02-15_59_15_CNN_1region_region_agnostic_Training_on_Interesting_Regions
    - 2026-01-31-13_19_18_CNN_1region_region_agnostic_input_reshaped_First_Try

- Addestro autoencoder con dati upsampled 

Noto Che vanno tutti benino <br>

Domande Marie:
- Selezione regioni: Effect size Cohen su power spectra?? Ha senso? (forse no ma per ora lo faccio)
- fai vedere visualization - momentaneo
- visualization - single subject - CNN only interesting regions vs CNN totale

## 05-06/02/2026
Selezione regioni con cohen based su power spectra è ok per fare benchmark, ma da migliorare nella pipeline finale!! (anche come coerenza)

Incontro con Marie DONE:
- prova a fare soggetto 8 (così così per localizzazione) e poi 1 (peggio) (più sparso nelle regioni)
    - PER QUESTI addestra (sia EEG che MEG)<br>
        '2026-01-30-18_41_52_Autoencoder_1region_region_agnostic_First_Try',<br>
        '2026-02-02-15_59_15_CNN_1region_region_agnostic_Training_on_Interesting_Regions',<br>
        '2026-01-31-13_19_18_CNN_1region_region_agnostic_input_reshaped_First_Try
    - e poi secondo step di addestramento con regioni scelte:
        - intersting all                                        (Done per tutti i soggetti)
        - cohen per subject (fai anche per soggetto 2)          (Done per tutti i soggetti)
        - cohen globale (fai anche per soggetto 2)              (Done per tutti i soggetti)
- Questo per verificare se:
    - per subject 2 avevamo visto che, aggregando su regioni Interesting avevamo:
        - CNN region agnostic su tutte le reigoni    era 0.71
        - CNN region agnostic su regioni interesting era 0.76
        - Autoencoder region agnostic era 0.73
    - quindi dobbiamo verificare se per gli altri soggetti capita stesso andamento
    - (o se classificazione aggregando Intersting regions non va bene perchè in questi soggetti sono altre le regioni ok!)

## 09-10/02/2026

- grafico di perfromances dei settings di giorni precedenti, per confrontare cosa succede al variare dei settings/soggetti.Vedi se trovi corrispondenze tra soggetti

## 11-12/02/2026

DONE:
- Addestra reti: stesse prove di <br>
        '2026-01-30-18_41_52_Autoencoder_1region_region_agnostic_First_Try',<br>
        '2026-02-02-15_59_15_CNN_1region_region_agnostic_Training_on_Interesting_Regions',<br>
        '2026-01-31-13_19_18_CNN_1region_region_agnostic_input_reshaped_First_Try <br>
    per i soggetti 1,2,8, ma con MEG 



## 13-15/02/2026:

- accorpo regioni (con relativo training) per prove che ho fatto il 11-12/02/2026 (MEG) e vedo che cambia (accorpo in 3 modi):
    - 'Yes Important Regions - all'
    - 'Global Selection - Cohen based'
    - "Cohen's d effect size selection" 


HO fatto il training (ACCORPATO):

- DataType  (EEG - MEG)
    - Subject (1, 2, 8)
        - Architecture (Autoencoder RA AllROI, CNN RA AllROI, CNN RA ImportantROI)
            - Region Selection (global cohen, Important, Cohen x subject)
                - Classifier (LDA, SVC, RandomForest)

## 16-18/02/2026:

16/02:<br>
Incontro tutti, TODO:

- Autoencoder first try, Before training SVC cohen based selection of features in bootleneck layer
- A valle della ri-nominazione delle regioni, standardizzato, ri-controlla guadagno tra autoencoders region agnostic all, region slected, region specific! 

In progress:

- Analizza risultati autoencoder region specific (sia performances quando aggrego, che reconstruction performances)

Done:
- Analizzato risultati autoencoder region agnostic cohen selected regions
- Autoencoder region agnostic cohen selected regions (Autoencoder_1region_region_agnostic_Training_on_8_cohen_selected)
- Autoencoder region specific cohen selected regions (Autoencoder_1region_region_specific_cohen_subject_specific)
- Autoencoder first try, now don't work on raw timeseries, but on frequency filtered timeseries (something like 6-14,12-30)

## 19-20/02/2026:
Incontro Marie: <br>
Posso dividere in 2 o 3 split le frequenze: alpha/low_beta/high_beta oppure alpha+low_beta/high_betas <br>

DONE IMPORTANTE:
- STANDARDIZZATO BENE NOTAZIONE REGIONI

## 23/02/2026:
addestro: 
- 2026-02-23-18_03_19_Autoencoder_1region_region_agnostic_separate_frequencies_4-25_20-45_Hz

## 24/02/2026
Studio: A key component of our convolutional block is the multi-receptive field (MRF) fusion, adapted from Kong et al. (2020) (Paper MultiBandAudio) <br>

!!!!!!!!!!!!!!!! ATTENTO: la divisione in 3 bande sembra essere molto più funzionale (a riprodurre timeseries ad alte freqs) !!!!!!!!!!!!!!!!! <br>

addestro: 
- 2026-02-24-14_14_08_Autoencoder_1region_region_agnostic_separate_frequencies_ConvTranspose4-25_20-45_Hz 
- 2026-02-24-15_33_41_Autoencoder_1region_region_agnostic_separate_frequencies_6-13_12-21_19-31_Hz

TODO - valuta aggiungere:
- residual blocks (Tommaso mi dice non è super ideale come cosa, dispendioso, serve a far convergere, prima e meglio, semplifica il compito ma non riduce la complessità del problema!)
- dilated convolution
- multi-receptive field

## 25/02/2026
addestro:
- 2026-02-27-10_37_22_Autoencoder_1region_region_agnostic_First_Try_No_BatchNorm
    - noto non migliora granchè


- sto provando a fare encoder timeseries, decoder power spectra (da fare)



## 26/02/2026
Incontro Marie <br>
TODO:

- Verificare se i non miglioramenti per soggetti 1,2,8 per autoencoder 3 bande, anche se fa meglio su alte freqs, sono giustificati dal fatto che tutti i soggetti indagati hanno freqs caratteristiche in alpha!!

    - A questo scopo, scelgo soggetti 4, 18 (attività in freq alte) confronta autoencoder "standard" vs autoencoder con bande splitted (3 bande freq) per vedere se qui miglioro la perfromance migliorando alte freqs!

- Prova a fare 3 normalizzazioni per decoder to PowerSpectra: (normalizzazione classica per regione; normalizzazione per banda; normalizzazione per bin di frequenza)

- OPPURE! non lavorare con i power spectra, ma con il loro logaritmo (così do maggiore importanza al peso delle alte freqs, ma evito di fare normalizzazioni strane)





## 27/02/2026
addestro:
- First try subj 4 e 18
- Autoencoder freqs divided in 3 bands

TODO:
- Introduci GRU layer da qualche parte dopo ultimo CONV in encoder

## 28/02/2026
Continuo cose

## 01/03/2026

- HO Controllato soggetti 4, 18 cosa succede se scelgo autoencoder Fist try (singola banda) e a 3 bande (addestra secondo ciclo, accorpo regioni)
    - noto per sbj 4 lieve miglioramento con 3 bande. Persubj 18 performance invariate tra i 2 casi


## 02/03/2026

- analizzo training di first try, NO batch norm per subj 1 e 2. Non ho cambiamenti rispetto performances di First try normale.


Incontro con tutti, TODO:
- Prova EEG to MEG (and vice-versa)
- addestro subj2 upsampled 5x (è una prova, vediamo)

## 03-05/03/2026

addestro:
- 2026-03-05-10_51_16_Encoder_timeseries_Decoder_spectra_log_simpleNormalization (sbj2 , 1, 4, 8)
- 2026-03-05-11_42_18_Encoder_timeseries_Decoder_spectra_log_simpleNormalization_Convolutional_Decoder (full convolutional in decoder per vedere che succede)

DONE:
- sto notando negli addestramenti che non ho "continuità" nello spettro ricostruito. Penso a questo punto di poter lasciare come ultimo layer un Conv1D (semmai padding=same dopo il dense? boh o iniziare già dall'inizio a fare qualcosa di mirato ad avere giusta dimensione boh vedi tu)
- ATTENTO in realtà la "non continuità" era nel power spectra vero, la ricostruzione è più smooth (see Spectra_reconstruced_03_idx7871 in this folder)

NOTO:
- ricostruzione è smooth. Significa che sto riuscendo a non imparare il rumore, ma colgo la natura del segnale?

- inizio a lavorare su EEG to MEG (da cambiare le funzioni con cui salvo i risultati)

TODO:
- Sto notado stranamente che cambiando totalmente struttura del decoder (FC to CONV) ottengo risultati quasi uguali! 
    - Non nelle metriche e performance (non ho fatto 2nd training) ma proprio nel ricostruire... Stranissimo!

## 06/03/2026

addestro:
- 2026-03-06-10_20_40_Autoencoder_1region_region_agnostic_EEG_to_MEG EEG to MEG (sbj 2 e 1)
    - noto non riesco proprio a riprodurre nessuna caratteristica della MEG partendo da EEG (ho fatto errori??? sembra di no)
- 2026-03-06-11_06_02_Encoder_timeseries_Decoder_spectra_log_simpleNormalization_larger_bandwidth_taper (bandwidth larger 8)
    - sembrerebbe leggermente meglio, almeno lo smoothing del power spectra è meglio. Le performance circa uguali (pochetto meglio per SVM).

## 10/03/2026
addestro:
- 2026-03-10-14_25_43_Autoencoder_1region_region_agnostic_First_Try_longer_windows (subj 1) (controllo se l'unico motivo delle performance migliori non è solamente perchè ho finestre temporali diverse e longer)
- 2026-03-10-15_14_58_Autoencoder_1region_region_agnostic_First_Try_GRU GRU!! (sbj 1,2,8)
- 2026-03-10-17_03_00_Autoencoder_1region_region_agnostic_First_Try_GRU_at_beginning (sbj 1, 2)



ATTENTO secondi training:<br>
-  (sbj 1) 2026-03-10-14_25_43_Autoencoder_1region_region_agnostic_First_Try_longer_windows per performance, ho verificato che non è la finestra più lunga ad aver fatto migliorare le performancs, rimangono uguali
- Ho inserito GRU "semplice" dopo ultimo conv encoder di Timeseries -> Timeseries,
    - Le performance non sembrano migliorare, ma il costo computazionale è 1.5x

fatto:
- sto addestrando CNN + GRU all'inizio, vedi se questo migliora rispetto a GRU alla fine 
    - (facendo secondo training accorpando regioni si vede nessuna differenza sostanziale) 

## 11/03/2026 
Incontro con tutti (il giorno dopo anche con Marie) <br>
TODO:
- vedi che succede al soggetto 8: tieni traccia del secondo preso in considerazione (se il soggetto sta facendo il task per esempio solo dopo un certo lasso di tempo)
    - Controlla performance vs tempo (vs MI e Rest)
    - media power spectra su trial vs tempo
    - differenza reconstructed vs originale al variare del tempo per MI e Rest
    - vedi output (confusion matrix) se non è tutto classificato come MI o Rest vs tempo (controllato, non è così)
        - ho verificato che le reti (per tutti i sbj) sono bilanciate nel numero di predizioni (MI vs Rest)
        - anche le confusion matrix sono abbastanza bilanciate, vedi perf vs time, % cambia ma con significato (se interpreto)

- Inizia fusione EEG MEG

DONE:
- Ho controllato EEG to MEG, che il problema che non ricostruisco nulla non fosse un prolema di scaling. Ho anche fatto l'inverso (MEG to EEG) e i risultati sono uguali
- ripeti analisi fatte fino ad ora (le più importanti) per MEG (da fare accorpare regioni)
- Inserisci gru in timeseries to powerspectra (potrebbe aiutare perchè ho finestre più grandi) (in realtà poco, non sensibilmente e non per tutti i sbj)
- Iniziato struttura Autoencoder+Classifier (non sembra essere meglio)

TODO:
- Struttura Autoencoder+Classifier: forse dovrei selezionare le regioni prima (per parte CNN serve?)



## 12/03/2026
addestro MEG:
- 2026-03-12-10_32_56_Encoder_timeseries_Decoder_spectra_log_simpleNormalization_bandwidth_taper_6 (sbjs 2, 1, 8)
- 2026-03-12-15_19_09_Autoencoder_1region_region_agnostic_MEG_to_EEG (sbj 1, 2)

EEG: 
- 2026-03-12-17_15_08_Encoder_timeseries_Decoder_spectra_log_simpleNormalization_bandwidth_taper_6_GRU_at_beginning (sbjs 1,2, 8)

## 13-17/03/2026
Inizio a costruire SupervisedAutoencoder
- sto pensando a come bilanciare le loss Reconstruction e Classification. (timeseries)
- Rec:      a inizio trainining sui 0.7, poi scende a 0.35 o 0.15 (in base a quanto riesco a riprodurre bene una regione)
- Class:    pred random loss è 0.7, poi tra 0.2 e 0.45 il minimo in base a quanto smoothing metto
- Class/3:  pred random loss è 0.23, poi tra 0.067 e 0.15 in base a quanto smoothing metto

Sclego per ora:
- smoothing 0.2
- binary_loss_weight = 1/3
(mostra prima lossWeight = 1, ragionamento su  pendenze, poi lossWeight = 0.333)

Addestro:
- 2026-03-13-13_13_21_SupervisedAutoencoder_1region_region_agnostic_First_Try_sm_0.20_classwei_0.33 (sbj 1,8, 2)
- 2026-03-13-19_35_05_Supervised_Encoder_timeseries_Decoder_spectra_log_simpleNormalization_bandwidth_taper_6_sm_0.20_classwei_0.25 (sbj 1, 2, 8, 4, 18)
- 2026-03-13-19_18_40_Supervised_Encoder_timeseries_Decoder_spectra_log_simpleNormalization_bandwidth_taper_6_sm_0.20_classwei_1 (sbj 1, 2, 8)


verifico cosa succede al sbj 8, perchè le performances sono strane.
- Per ora seguo i test stabiiti giorno 11-12/03, ma ancora nessun indizio
- noto che performance per fold è molto variabile e che tra i diversi fold ho regioni diverse selezionate. Che sia questo che influenza la variabilità?
    - provo a fare stesso addestramento ma avendo una selezione di regioni fissa per tutti i fold
        - non miglioro interpretazione granchè. Se scelgo regioni globali cohen, sembra che il soggetto faccia meglio solo nei primi windows

## 18-20/03/2026


### Osservazioni interessanti performance vs tempo:

Noto che per soggetto 2, qualsiasi tipo di selezione delle regioni faccia (accorpo secondo training, sia fold wise selection or fixed), la performance al variare del tempo ha andamento: <br>
0.75 -> 0.85 -> 0.8 <br>
(molto bene, inizia un po' dopo, termina un po' prima!). Questo spiega anche il calo un po' di performance rispetto al caso in cui usavo le features estratte a mano dai PowerSpectra, perchè utilizzavo l'intera finestra temporale! 

Per soggetto 1, performance sono sostanzialmente migliori se faccio selezione cohen_subject_specific. In questo caso sembra fare benissimo il task inizialmente e poi lasciarsi andare un pochetto (nel soggetto 1 se scelgo regioni motorie non ho nessuna performance sensibilmente diversa da chance)

Per soggetto 8, ho interpretazione (debole) che il task viene fatto all'inizio e poi non più. 

OSSERVAZIONI IMPORTANTI:<br>
Ragionando insieme a Marie e Silvia concludiamo che: 
- esistono alcuni sbj (2) chiaramente "neurotipici" prevedibili, tutto ok (regioni giuste (ok), feature giuste (funzionava anche online o con handextracted di powerspectra analisi mia di prima), regioni robuste)
- alcuni sbj (1) che hanno le loro firme neaurali tipiche ma "forti, persistenti", coerenti nelle strategie, ma "particolari,singolari" quindi online non va bene con hand-extracted features in maniera blind (regioni sono diverse, features sono diverse (online una chiavica, con handextracted era non molto buono, ma autoencoder cool), regioni robuste)
- altri sbj (8) non hanno firme chiare, regioni non chiare, robuste per selezione di regioni cohen


TODO: 
- Vorrei provare a trovare un modo (con features estratte con Deep) per classificare tutta la traccia invece che la singola window piccolina

DONE: 
- prova ad usare train_single_model_investigate2, in cui per x_train scegli dati a partire da 3 secondi, ma per x_val tutti anche prima, per vedere che faceva la gente prima che comparisse il feedback
- prova ad investigare separatamente MI e Rest (performances across time, avendo addestrato solo iniziando da 3 secondi) (se Rest è 100% performance e MI 0%, pei questi soggetti POTREBBERO star dormendo prima )


### Incontro con tutti (no Pierpa)

Addestra (Check): 
- sia power spectra che timeseries
    - sia autoencoder easy che Supervised autoencoder 
        - region agnostic solo su cohen selected per subject
        - region specific 1 region
        
- Cerca di fare alla fine MEG + EEG (vedi con quale architettura)

## 23-27/03/2026

Fatti in precedenza (anche secondo training):
- Autencoder Timeseries region agnostic cohen (2026-02-16-18_25_42_Autoencoder_1region_region_agnostic_Training_on_8_cohen_selected)
- Autencoder Timeseries region specific cohen x subject

Addestro (TODO fai secondo training, accorpa regioni): 
- 2026-03-23-18_07_35_Encoder_timeseries_Decoder_spectra_log_simpleNormalization_region_specific_cohen_per_subj(sbj 1,2,8) (1,2,8)
- 2026-03-23-20_54_16_Encoder_timeseries_Decoder_spectra_log_simpleNormalization_cohenXsubj (sbj 1,2,8,4,18) (1,2,8, 4,18)


2026-03-24-11_30_34_Supervised_Encoder_timeseries_Decoder_spectra_region_agnostic_selected_cohen_per_fold_log_simpleNormalization_taper_6_sm_0.20_classwei_0.25
- 2026-03-24-11_48_10_Supervised_Encoder_timeseries_Decoder_spectra_region_agnostic_selected_cohen_per_fold_log_simpleNormalization_taper_6_sm_0.20_classwei_0.25 (sbj 1,2,8,4,18) (sbj 1,2,4,8,18) 
########################DA QUI#######################
- 2026-03-24-12_22_53_SupervisedAutoencoder_1region_agnostic_selected_cohen_per_fold_sm_0.20_classwei_0.33 (sbj 1,2,4,8,18)
- 2026-03-24-12_39_44_Supervised_Encoder_timeseries_Decoder_spectra_region_specific_log_simpleNormalization_taper_6_sm_0.20_classwei_0.25 (sbj 1,2,4,8)
- 2026-03-24-14_19_51_SupervisedAutoencoder_1region_region_specific_sm_0.20_classwei_0.33 (1,2,4)


TODO DA ADDESTRARE:
- SupervisedAutoencoder_1region_region_specific (18)
- SupervisedEncoder_1region_region_specific (sbj 18)


## 31/03/2026

Addestro:
- 2026-03-31-13_27_35_Encoder_timeseries_Decoder_spectra_log_simpleNormalization_EEG_MEG_GRU_at_beginning (sbj 1)
- 2026-03-31-14_15_41_Encoder_timeseries_Decoder_spectra_log_simpleNormalization_EEG_MEG (sbj 1, 2, 8)

# Access resources

- cluster ICM (I think Access available only from workstation desktop ??)
- workstation Dell office, ask Arthutr (VPN ? https://myicm.icm-institute.org/accessing-telecommuting-resources)